# Qwen3-VL-4B joint classification + generation (WeldMind)

This notebook fine-tunes **Qwen3-VL-4B** on RIAWELC weld-defect radiographs to
test the hypothesis that adding a descriptive-caption auxiliary objective
improves classification accuracy in the small-data regime.

**Sweep**: 3 subset sizes × 3 loss variants × 1 seed = **9 runs**, evaluated on
the real RIAWELC validation split (6,102 imgs). See `README.md` for the full
description.

Run cells top-to-bottom on a single H20-NVLink (96 GB). Wall-clock ~4.5 h.

## Cell 1 — Setup, paths, GPU sanity

In [ ]:
import os, sys, time, json, subprocess
from pathlib import Path

# Make the helper package importable when the kernel cwd is the notebooks dir.
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    # Fall back if cwd is the repo root.
    NOTEBOOK_DIR = (NOTEBOOK_DIR / "notebooks").resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

import torch
from weldmind_train.config import (
    PROJECT_ROOT, REPO_DATA_DIR, RIAWELC_FULL_DIR, RIAWELC_RAR_DIR,
    MODEL_CACHE_DIR, RUNS_DIR, SPLITS_DIR,
    MODEL_ID_PRIMARY, MODEL_ID_FALLBACK,
    LABEL_LIST, RUN_CONFIG,
)

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
    print("mem total:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

print("\nproject root:", PROJECT_ROOT)
print("data dir:", REPO_DATA_DIR)
print("RIAWELC full (will be created on extract):", RIAWELC_FULL_DIR)
print("RAR archives at:", RIAWELC_RAR_DIR)
print("model cache:", MODEL_CACHE_DIR)
print("runs dir:", RUNS_DIR)

RUNS_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(RUN_CONFIG["seed"])
print("seed:", RUN_CONFIG["seed"])

## Cell 2 — Extract official RIAWELC val/test from RAR

Calls `preprocessing/data/rar_extractor.py` to unpack the 19 .partNN.rar files
into `data/riawelc_full/`. Idempotent — skips if validation/ already exists.
After extraction, asserts the expected layout and prints class counts.

In [ ]:
from weldmind_train.config import RIAWELC_FULL_DIR, RIAWELC_RAR_DIR

def _find_split_dir(base: Path, split: str):
    for cand in (base / "DB" / split / "collect", base / split, base / split / "collect"):
        if cand.is_dir():
            return cand
    return None

val_dir = _find_split_dir(RIAWELC_FULL_DIR, "validation")
test_dir = _find_split_dir(RIAWELC_FULL_DIR, "testing")

if val_dir is None or test_dir is None:
    extractor = PROJECT_ROOT / "preprocessing" / "data" / "rar_extractor.py"
    if not extractor.exists():
        raise FileNotFoundError(extractor)
    print(f"extracting {RIAWELC_RAR_DIR} → {RIAWELC_FULL_DIR}")
    proc = subprocess.run(
        [sys.executable, str(extractor),
         "--rar-dir", str(RIAWELC_RAR_DIR),
         "--out-dir", str(RIAWELC_FULL_DIR)],
        capture_output=True, text=True,
    )
    print("stdout:", proc.stdout[-2000:])
    print("stderr:", proc.stderr[-2000:])
    if proc.returncode != 0:
        raise RuntimeError(f"rar extraction failed (exit {proc.returncode})")

    val_dir = _find_split_dir(RIAWELC_FULL_DIR, "validation")
    test_dir = _find_split_dir(RIAWELC_FULL_DIR, "testing")

assert val_dir is not None, f"validation/ not found under {RIAWELC_FULL_DIR}"
assert test_dir is not None, f"testing/ not found under {RIAWELC_FULL_DIR}"
print("validation:", val_dir)
print("testing:   ", test_dir)

for label_dir in (val_dir, test_dir):
    counts = {p.name: len(list(p.glob("*.png"))) for p in sorted(label_dir.iterdir()) if p.is_dir()}
    print(f"  {label_dir.parent.name}:", counts, "total =", sum(counts.values()))

## Cell 3 — Download Qwen3-VL-4B from ModelScope

Tries the Instruct id first, then the base id. Cached at `~/models/`.

In [ ]:
from modelscope import snapshot_download

def _snapshot(model_id: str):
    try:
        path = snapshot_download(model_id, cache_dir=str(MODEL_CACHE_DIR))
        return Path(path), model_id
    except Exception as e:
        print(f"  {model_id} → {type(e).__name__}: {e}")
        return None, None

MODEL_PATH, MODEL_ID = _snapshot(MODEL_ID_PRIMARY)
if MODEL_PATH is None:
    print("falling back to", MODEL_ID_FALLBACK)
    MODEL_PATH, MODEL_ID = _snapshot(MODEL_ID_FALLBACK)

assert MODEL_PATH is not None, "Could not download either Qwen3-VL-4B variant from ModelScope. " \
    "Confirm internet access and that the model id exists on the ModelScope hub."

print("model id:", MODEL_ID)
print("model path:", MODEL_PATH)
print("files:", sorted(p.name for p in MODEL_PATH.iterdir())[:15])

## Cell 4 — Load training pool

Joins on-disk `data/train_batch/` with `data/train_results_merged.json` to
produce a list of `RiawelcExample` records (each with image_path, label,
class_idx, description). Drops any image whose description is missing.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

from weldmind_train.data import build_train_pool

TRAIN_POOL = build_train_pool()
print(f"\ntotal: {len(TRAIN_POOL)} examples")
from collections import Counter
print("class distribution:", Counter(ex.class_idx for ex in TRAIN_POOL))
print("\nfirst example:")
print("  image:", TRAIN_POOL[0].image_path.name)
print("  label:", TRAIN_POOL[0].label)
print("  description[:200]:", TRAIN_POOL[0].description[:200])

## Cell 5 — Stratified subset sampler

Builds the {50, 200, 1000}-per-class subsets we'll sweep over.

In [ ]:
from weldmind_train.data import stratified_subset

SUBSETS = {}
for n in RUN_CONFIG["subset_sizes_per_class"]:
    SUBSETS[n] = stratified_subset(TRAIN_POOL, n, seed=RUN_CONFIG["seed"])
    classes = Counter(ex.class_idx for ex in SUBSETS[n])
    print(f"  per_class={n:5d}  total={len(SUBSETS[n]):5d}  classes={dict(classes)}")

## Cell 6 — Processor + collator smoke test

Loads the Qwen3-VL processor and runs one tiny batch through the collator so
we can eyeball `input_ids`, `labels`, and `loss_weight` alignment before
burning GPU time. Prints token counts for prompt/label/description regions
and confirms `-100` masking is in the right places.

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(str(MODEL_PATH), trust_remote_code=True)
print("processor:", type(processor).__name__)
print("tokenizer:", type(processor.tokenizer).__name__)
print("eos:", processor.tokenizer.eos_token, "pad:", processor.tokenizer.pad_token)

from weldmind_train.collator import Qwen3VLCollator

smoke_batch = [SUBSETS[50][0], SUBSETS[50][1]]
for variant in ("V_cls", "V_joint", "V_weighted"):
    col = Qwen3VLCollator(
        processor=processor, variant=variant,
        lambda_label=RUN_CONFIG["lambda_label_weighted"],
    )
    out = col(smoke_batch)
    n_total = out["input_ids"].numel()
    n_train = (out["labels"] != -100).sum().item()
    n_w_label = (out["loss_weight"] > 1.0).sum().item()
    n_w_one = ((out["loss_weight"] > 0) & (out["loss_weight"] <= 1.0 + 1e-6)).sum().item()
    print(f"  {variant:11s}  total={n_total}  trainable={n_train}  "
          f"weighted(>1)={n_w_label}  weight==1={n_w_one}  "
          f"pixel_values={tuple(out['pixel_values'].shape) if 'pixel_values' in out else None}")

## Cell 7 — Load base model, freeze vision, attach LoRA

Sets up a fresh PEFT-wrapped model. We reload this from disk at the start of
every run in the sweep (cell 8) so adapters don't bleed between variants.

In [ ]:
from transformers import AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

def _freeze_vision_tower(model):
    """Freeze any submodule with vision in its name. Qwen-VL uses `visual`."""
    n_frozen = 0
    for name, p in model.named_parameters():
        if any(k in name for k in (".visual.", "vision_tower", "vision_model.", "visual_model")):
            p.requires_grad_(False)
            n_frozen += p.numel()
    return n_frozen

def load_fresh_lora_model():
    """Reloads base model + reattaches a fresh LoRA adapter. Call once per run."""
    model = AutoModelForImageTextToText.from_pretrained(
        str(MODEL_PATH),
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="cuda:0",
        trust_remote_code=True,
    )

    frozen = _freeze_vision_tower(model)
    print(f"  froze vision params: {frozen/1e6:.1f}M")

    lc = RUN_CONFIG["lora"]
    lora_cfg = LoraConfig(
        r=lc["r"], lora_alpha=lc["alpha"], lora_dropout=lc["dropout"],
        target_modules=lc["target_modules"],
        task_type=TaskType.CAUSAL_LM, bias="none",
    )
    model = get_peft_model(model, lora_cfg)
    model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    model.print_trainable_parameters()
    return model

# Quick smoke load (we'll throw it away — sweep reloads fresh per run).
_smoke_model = load_fresh_lora_model()
print("\npeak GPU mem after load:", round(torch.cuda.max_memory_allocated()/1024**3, 2), "GB")
del _smoke_model
torch.cuda.empty_cache()

## Cell 8 — Sweep: train 9 adapters

Nested loop: 3 subset sizes × 3 loss variants × 1 seed. Each run gets a
fresh model + fresh `WeightedCETrainer`; the adapter is saved under
`runs/n{size}_v{variant}/`. Epoch budget scales inversely with subset size.

In [ ]:
from torch.utils.data import Dataset
from transformers import TrainingArguments
from weldmind_train.trainer import WeightedCETrainer

class ListDataset(Dataset):
    def __init__(self, records): self.records = records
    def __len__(self): return len(self.records)
    def __getitem__(self, i): return self.records[i]

def run_one(subset_size: int, variant: str):
    out_dir = RUNS_DIR / f"n{subset_size}_v{variant}"
    if (out_dir / "adapter_model.safetensors").exists():
        print(f"  [skip] adapter already exists: {out_dir}")
        return out_dir

    records = SUBSETS[subset_size]
    epochs = RUN_CONFIG["epochs_by_size"][subset_size]
    ds = ListDataset(records)

    collator = Qwen3VLCollator(
        processor=processor, variant=variant,
        lambda_label=RUN_CONFIG["lambda_label_weighted"],
    )

    model = load_fresh_lora_model()
    args = TrainingArguments(
        output_dir=str(out_dir),
        per_device_train_batch_size=RUN_CONFIG["per_device_train_batch_size"],
        gradient_accumulation_steps=RUN_CONFIG["gradient_accumulation_steps"],
        num_train_epochs=epochs,
        learning_rate=RUN_CONFIG["learning_rate"],
        lr_scheduler_type=RUN_CONFIG["lr_scheduler"],
        warmup_ratio=RUN_CONFIG["warmup_ratio"],
        weight_decay=RUN_CONFIG["weight_decay"],
        logging_steps=RUN_CONFIG["logging_steps"],
        bf16=True, tf32=True,
        save_strategy="no",
        report_to=[],
        remove_unused_columns=False,
        dataloader_num_workers=2,
        seed=RUN_CONFIG["seed"],
    )

    trainer = WeightedCETrainer(
        model=model, args=args, train_dataset=ds, data_collator=collator,
    )
    t0 = time.time()
    trainer.train()
    dt = time.time() - t0

    out_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(out_dir))
    processor.save_pretrained(str(out_dir))
    (out_dir / "meta.json").write_text(json.dumps({
        "subset_size": subset_size, "variant": variant,
        "epochs": epochs, "train_time_sec": dt,
        "n_train_examples": len(records),
        "model_id": MODEL_ID, "seed": RUN_CONFIG["seed"],
    }, indent=2))

    del trainer, model
    torch.cuda.empty_cache()
    print(f"  [done] {subset_size:5d}/{variant} in {dt/60:.1f} min → {out_dir}")
    return out_dir

ALL_RUNS = []
for size in RUN_CONFIG["subset_sizes_per_class"]:
    for variant in RUN_CONFIG["variants"]:
        print(f"\n=== run: subset={size}/class, variant={variant} ===")
        ALL_RUNS.append((size, variant, run_one(size, variant)))

print("\nfinished", len(ALL_RUNS), "runs.")

## Cell 9 — Evaluate every adapter on the real validation split

Loads each saved adapter on top of the base model, runs greedy decoding on
the 6,102-image validation set, parses the first label string from each
generation, and writes per-run metrics to `runs/results.csv`.

In [ ]:
import pandas as pd
from peft import PeftModel
from weldmind_train.data import build_eval_records
from weldmind_train.eval import evaluate_classification

EVAL_RECORDS = build_eval_records("validation")
print(f"eval set: {len(EVAL_RECORDS)} images")

def _load_eval_model(adapter_dir: Path):
    base = AutoModelForImageTextToText.from_pretrained(
        str(MODEL_PATH),
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
        device_map="cuda:0",
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base, str(adapter_dir))
    model.eval()
    return model

rows = []
for size, variant, adapter_dir in ALL_RUNS:
    print(f"\n--- eval: n={size}/cls v={variant} ---")
    model = _load_eval_model(adapter_dir)
    metrics = evaluate_classification(
        model, processor, EVAL_RECORDS,
        max_new_tokens=RUN_CONFIG["max_new_tokens_eval"],
        desc=f"n{size}_v{variant}",
    )
    print(f"  acc={metrics['accuracy']:.4f}  macroF1={metrics['macro_f1']:.4f}  "
          f"parse_failed={metrics['n_parse_failed']}")
    rows.append({
        "subset_size_per_class": size,
        "variant": variant,
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        **{f"f1_{cls}": v for cls, v in zip(LABEL_LIST, metrics["per_class_f1"])},
        "n_parse_failed": metrics["n_parse_failed"],
        "n_total": metrics["n_total"],
        "adapter": str(adapter_dir),
    })
    del model
    torch.cuda.empty_cache()

results_df = pd.DataFrame(rows)
results_df.to_csv(RUNS_DIR / "results.csv", index=False)
print("\nwrote", RUNS_DIR / "results.csv")
results_df

## Cell 10 — Plot accuracy vs subset size, one line per variant

Answers the research question visually: does V_joint (and V_weighted) beat
V_cls at small N, and by how much? If yes, the auxiliary generation
objective helps the model in the low-data regime.

In [ ]:
import matplotlib.pyplot as plt

pivot_acc = results_df.pivot(index="subset_size_per_class",
                              columns="variant", values="accuracy")
pivot_f1 = results_df.pivot(index="subset_size_per_class",
                             columns="variant", values="macro_f1")

print("\naccuracy:")
print(pivot_acc)
print("\nmacro F1:")
print(pivot_f1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for variant in RUN_CONFIG["variants"]:
    if variant in pivot_acc.columns:
        axes[0].plot(pivot_acc.index, pivot_acc[variant], marker="o", label=variant)
        axes[1].plot(pivot_f1.index, pivot_f1[variant], marker="o", label=variant)
for ax, ylab in zip(axes, ("accuracy", "macro F1")):
    ax.set_xscale("log"); ax.set_xticks(RUN_CONFIG["subset_sizes_per_class"])
    ax.set_xticklabels(RUN_CONFIG["subset_sizes_per_class"])
    ax.set_xlabel("training images per class"); ax.set_ylabel(ylab)
    ax.grid(alpha=0.3); ax.legend()
fig.suptitle("Joint vs cls-only fine-tuning of Qwen3-VL-4B on RIAWELC validation")
fig.tight_layout()
fig.savefig(RUNS_DIR / "sweep_plot.png", dpi=150)
plt.show()

print("\nNEXT: rerun cell 9 with build_eval_records('testing') for the paper-table numbers on the 2,441-image test split.")